In [ ]:
!pip install -q librosa scipy pandas openpyxl gdown matplotlib
!pip install -q transformers
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install pytorch-tcn

In [ ]:
!git clone https://github.com/alexandarM/AI-SPEAK_catB.git

Cloning into 'AI-SPEAK_catB'...
remote: Enumerating objects: 234, done.
remote: Counting objects: 100% (234/234), done.
remote: Compressing objects: 100% (166/166), done.
remote: Total 234 (delta 130), reused 167 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (234/234), 91.00 KiB | 1.42 MiB/s, done.
Resolving deltas: 100% (130/130), done.


In [ ]:
%cd AI-SPEAK_catB

/content/AI-SPEAK_catB


In [ ]:
!python download_data.py

Downloading...
From (original): https://drive.google.com/uc?id=1AqENkQjMEC4BLUVaaXIueK4q-KwQyD7S
From (redirected): https://drive.google.com/uc?id=1AqENkQjMEC4BLUVaaXIueK4q-KwQyD7S&confirm=t&uuid=bfabb983-33cd-4b50-8a2d-e9a5e0784e88
To: /content/AI-SPEAK_catB/data/spk14_blendshapes.zip
100% 59.4M/59.4M [00:01<00:00, 36.5MB/s]
Unzipping spk14_blendshapes.zip
Downloading...
From (original): https://drive.google.com/uc?id=1onjc_aw990qQAsvszLSI9rpXS2SkJFVd
From (redirected): https://drive.google.com/uc?id=1onjc_aw990qQAsvszLSI9rpXS2SkJFVd&confirm=t&uuid=31c39a3a-eb0f-43b0-9e5a-3eaf22d1aa2d
To: /content/AI-SPEAK_catB/data/spk08_blendshapes.zip
100% 67.9M/67.9M [00:01<00:00, 42.4MB/s]
Unzipping spk08_blendshapes.zip
Downloading...
From: https://drive.google.com/uc?id=1KduEMO5DqqdQLXPd45W6knSFIuODxMqR
To: /content/AI-SPEAK_catB/data/labels_aligned.zip
100% 278k/278k [00:00<00:00, 4.26MB/s]
Unzipping labels_aligned.zip
Downloading...
From (original): https://drive.google.com/uc?id=16e8VnAhH2L-

In [ ]:
from src.utils.dataset import BlendshapeDataset, collate_fn_mfcc, collate_fn_hubert
from src.config import BLENDSHAPE_NAMES
from torch.utils.data import DataLoader

DATA_ROOT = 'data'
ds = BlendshapeDataset(DATA_ROOT, speakers=['spk08', 'spk14'], load_synth=False, use_preprocessing=True, hubert_dir=None)


[Dataset] 358 real + 0 synth = 358 total samples


In [ ]:
from genericpath import exists
result_dict = "/content/results_final"
import os
os.mkdir(result_dict)

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from scipy.signal import savgol_filter

from src.preprocessing.audio import preprocess_waveform, resample_for_hubert
from src.preprocessing.features import extract_audio_features
from src.models.tcn import BlendshapeTCN


TEST_DIR   = "/content/AI-SPEAK_catB/data/test_set_catB/test_set_catB"
CKPT_PATH  = "/content/best_model_tcn_mfcc.pt"
SUBMIT_DIR = "submission_final"
FPS        = 60

os.makedirs(SUBMIT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


ckpt   = torch.load(CKPT_PATH, map_location=device)
config = ckpt.get("config", {})

model = BlendshapeTCN(
    audio_type   = "mfcc",
    d_model      = config.get("d_model", 256),
    use_phonemes = config.get("use_phonemes", True),
    n_layers     = config.get("n_layers", 5),
).to(device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Model učitan — epoha {ckpt.get('epoch','?')}  val_loss={ckpt.get('val_loss', float('nan')):.5f}")

test_wavs = sorted([
    os.path.join(TEST_DIR, f)
    for f in os.listdir(TEST_DIR)
    if f.lower().endswith(".wav")
])
print(f"Pronađeno {len(test_wavs)} fajlova.")

all_meta = {}

for wav_path in test_wavs:
    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    out_name  = f"{base_name}_result"
    print(f"\n{'='*55}")
    print(f"Obrada: {base_name}.wav")


    audio_duration_sec = sf.info(wav_path).duration

    t0 = time.perf_counter()

    audio    = preprocess_waveform(wav_path)   # mono + normalizacija
    n_frames = int(audio_duration_sec * FPS)

    af = extract_audio_features(wav_path, n_frames, use_preprocessing=True)
    af_t = torch.from_numpy(af).unsqueeze(0).to(device)   # (1, T, D_mfcc)

    pi = torch.zeros((1, n_frames),      dtype=torch.long,    device=device)
    pt = torch.zeros((1, n_frames, 1),   dtype=torch.float32, device=device)
    si = torch.zeros((1, n_frames),      dtype=torch.long,    device=device)
    ln = torch.tensor([n_frames],        dtype=torch.long,    device=device)

    with torch.no_grad():
        pred = model(
            af=af_t, pi=pi, pt=pt, si=si, lengths=ln
        ).squeeze(0).cpu().numpy()   # (T, 52)

    inference_time_sec = time.perf_counter() - t0

    pred_smooth = savgol_filter(pred, window_length=7, polyorder=2, axis=0)
    pred_smooth = np.clip(pred_smooth, 0.0, 1.0).astype(np.float32)


    rtf          = inference_time_sec / audio_duration_sec
    lookahead_ms = 0

    print(f"  Trajanje:           {audio_duration_sec:.3f} s")
    print(f"  Frejmovi:           {n_frames}")
    print(f"  inference_time_sec: {inference_time_sec:.4f} s")
    print(f"  RTF:                {rtf:.4f}  ({'✓ real-time' if rtf < 1 else '✗ sporije'})")
    print(f"  lookahead_ms:       {lookahead_ms:.1f} ms")


    csv_path = os.path.join(SUBMIT_DIR, f"{out_name}.csv")
    pd.DataFrame(pred_smooth).to_csv(
        csv_path, index=False, header=False, float_format="%.6f"
    )
    print(f"  → {csv_path}")

    all_meta[out_name] = {
        "inference_time_sec": round(inference_time_sec, 4),
        "audio_duration_sec": round(audio_duration_sec, 4),
        "rtf":                round(rtf, 6),
        "lookahead_ms":       round(lookahead_ms, 1),
        "fps_out":            FPS,
        "n_frames":           int(pred_smooth.shape[0]),
    }


meta_path = os.path.join(SUBMIT_DIR, "meta.json")
with open(meta_path, "w") as f:
    json.dump(all_meta, f, indent=2)

print(f"\n{'='*55}")
print(f"{'Fajl':<30} {'inf(s)':>8} {'RTF':>8}")
print("-" * 55)
for name, m in all_meta.items():
    flag = "✓" if m["rtf"] < 1 else "✗"
    print(f"{name:<30} {m['inference_time_sec']:>8.4f} {m['rtf']:>7.4f}{flag}")
print(f"\nSačuvano u: {os.path.abspath(SUBMIT_DIR)}")
print(json.dumps(all_meta, indent=2))


Device: cpu
TCN: 5 slojeva, RF=125 frejmova (2083ms)
Parametara: 3,145,764
Model učitan — epoha 78  val_loss=0.00539
Pronađeno 12 fajlova.

Obrada: Jovana1.wav
  Trajanje:           4.490 s
  Frejmovi:           269
  inference_time_sec: 0.3074 s
  RTF:                0.0685  (✓ real-time)
  lookahead_ms:       0.0 ms
  → submission_final/Jovana1_result.csv

Obrada: Jovana5.wav
  Trajanje:           3.715 s
  Frejmovi:           222
  inference_time_sec: 0.2267 s
  RTF:                0.0610  (✓ real-time)
  lookahead_ms:       0.0 ms
  → submission_final/Jovana5_result.csv

Obrada: Sinisa1.wav
  Trajanje:           5.234 s
  Frejmovi:           314
  inference_time_sec: 0.3272 s
  RTF:                0.0625  (✓ real-time)
  lookahead_ms:       0.0 ms
  → submission_final/Sinisa1_result.csv

Obrada: Sinisa7.wav
  Trajanje:           4.424 s
  Frejmovi:           265
  inference_time_sec: 0.1857 s
  RTF:                0.0420  (✓ real-time)
  lookahead_ms:       0.0 ms
  → submission_fi